# 06 — Segment Analysis

Aggregate results can hide important patterns. Segment analysis asks:

- Does the treatment effect hold across all user groups?
- Is the effect driven by a specific subgroup?
- Are there groups being harmed while the average looks neutral?

This notebook covers:

1. **Temporal analysis** — does the effect change over time? (novelty effect)
2. **Day-of-week analysis** — is behavior consistent across the week?
3. **Simpson's Paradox** — how aggregation can reverse or mask effects
4. **Interaction effects** — formally testing heterogeneous treatment effects
5. **Practical guardrails** — pre-specification and multiple testing preview

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.append("../src")
from ab_stats import two_proportion_ztest, confidence_interval

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

ALPHA = 0.05

df = pd.read_csv("../data/ab_data_clean.csv", parse_dates=["timestamp"])
df["weekday"]  = df["timestamp"].dt.day_name()
df["week_num"] = (df["timestamp"] - df["timestamp"].min()).dt.days // 7 + 1
df["is_weekend"] = df["weekday"].isin(["Saturday", "Sunday"])

print(f"Date range: {df.timestamp.min().date()} to {df.timestamp.max().date()}")
print(f"Weeks in experiment: {df.week_num.max()}")

## 1. Temporal Analysis — Novelty Effect

When a new design launches, users may engage with it simply because it's new.
This **novelty effect** can inflate early results. If the treatment effect shrinks
in later weeks, that's a sign you're measuring novelty, not genuine improvement.

Conversely, a **learning effect** means users improve at converting as they get used
to the new design — the effect grows over time.

In [ ]:
weekly = (
    df.groupby(["week_num", "group"])["converted"]
    .agg(conversions="sum", users="count")
    .reset_index()
)
weekly["rate"] = weekly["conversions"] / weekly["users"]

fig, ax = plt.subplots(figsize=(9, 4))
for group, color in [("control", "steelblue"), ("treatment", "coral")]:
    sub = weekly[weekly["group"] == group]
    ax.plot(sub["week_num"], sub["rate"] * 100, marker="o",
            color=color, linewidth=2, label=group.title())
    ax.fill_between(sub["week_num"], sub["rate"]*100 - 0.3, sub["rate"]*100 + 0.3,
                    color=color, alpha=0.1)

ax.set_xlabel("Week of experiment")
ax.set_ylabel("Conversion rate (%)")
ax.set_title("Weekly conversion rates by group — checking for novelty effect")
ax.legend()
plt.tight_layout()
plt.show()

# Weekly lift
print("Week | Control | Treatment | Lift")
print("-" * 42)
for w in sorted(weekly["week_num"].unique()):
    wk = weekly[weekly["week_num"] == w]
    rc = wk[wk["group"]=="control"]["rate"].values
    rt = wk[wk["group"]=="treatment"]["rate"].values
    if len(rc) and len(rt):
        print(f"  {w}  | {rc[0]:.4%}  | {rt[0]:.4%}     | {rt[0]-rc[0]:+.4%}")

## 2. Day-of-Week Analysis

User behavior varies by day. A test that runs Mon–Thu might show different results
than one running Fri–Sun. We should verify the effect is consistent across all days.

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

daily = (
    df.groupby(["weekday", "group"])["converted"]
    .agg(conversions="sum", users="count")
    .reset_index()
)
daily["rate"] = daily["conversions"] / daily["users"]
daily["weekday"] = pd.Categorical(daily["weekday"], categories=day_order, ordered=True)
daily = daily.sort_values("weekday")

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(7)
w = 0.35
for i, (group, color) in enumerate([("control","steelblue"),("treatment","coral")]):
    sub = daily[daily["group"]==group].set_index("weekday").reindex(day_order)
    ax.bar(x + i*w, sub["rate"]*100, width=w, label=group.title(), color=color, alpha=0.85)

ax.set_xticks(x + w/2)
ax.set_xticklabels([d[:3] for d in day_order])
ax.set_ylabel("Conversion rate (%)")
ax.set_title("Conversion rate by day of week")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Device Type Segment Analysis

Our dataset includes `device_type` (Mobile / Desktop / Tablet). Does the treatment
effect hold across all devices, or is it driven by one segment?

This is a pre-specified segment (device type is known at visit time), so we test
at the nominal α without correction.

In [ ]:
dev_seg = (
    df.groupby(["device_type", "group"])["converted"]
    .agg(conversions="sum", users="count")
    .reset_index()
)
dev_seg["rate"] = dev_seg["conversions"] / dev_seg["users"]

devices = df["device_type"].unique()
print("Device Type | Control Rate | Treatment Rate | Lift    | Significant?")
print("-" * 72)
for dev in sorted(devices):
    sub = dev_seg[dev_seg["device_type"] == dev]
    rc = sub[sub["group"]=="control"]["rate"].values[0]
    rt = sub[sub["group"]=="treatment"]["rate"].values[0]
    nc = sub[sub["group"]=="control"]["users"].values[0]
    nt = sub[sub["group"]=="treatment"]["users"].values[0]
    cc = sub[sub["group"]=="control"]["conversions"].values[0]
    ct = sub[sub["group"]=="treatment"]["conversions"].values[0]
    _, p, _, _, la, _ = two_proportion_ztest(cc, nc, ct, nt)
    sig = "Yes *" if p < ALPHA else "No"
    print(f"{dev:<12} | {rc:>12.2%} | {rt:>14.2%} | {la:>+7.2%} | {sig}")

# Visualization
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(sorted(devices)))
w = 0.35
for i, (group, color) in enumerate([("control","steelblue"),("treatment","coral")]):
    sub = dev_seg[dev_seg["group"]==group].sort_values("device_type")
    ax.bar(x + i*w, sub["rate"]*100, width=w, label=group.title(), color=color, alpha=0.85)
ax.set_xticks(x + w/2)
ax.set_xticklabels(sorted(devices))
ax.set_ylabel("Conversion rate (%)")
ax.set_title("Conversion rate by device type")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Gender Segment Analysis

Does the treatment effect differ by gender? Heterogeneous treatment effects here
could indicate the design resonates differently with different audiences.

In [ ]:
gen_seg = (
    df.groupby(["gender", "group"])["converted"]
    .agg(conversions="sum", users="count")
    .reset_index()
)
gen_seg["rate"] = gen_seg["conversions"] / gen_seg["users"]

genders = sorted(df["gender"].unique())
print("Gender | Control Rate | Treatment Rate | Lift    | Significant?")
print("-" * 68)
for g in genders:
    sub = gen_seg[gen_seg["gender"] == g]
    rc = sub[sub["group"]=="control"]["rate"].values[0]
    rt = sub[sub["group"]=="treatment"]["rate"].values[0]
    nc = sub[sub["group"]=="control"]["users"].values[0]
    nt = sub[sub["group"]=="treatment"]["users"].values[0]
    cc = sub[sub["group"]=="control"]["conversions"].values[0]
    ct = sub[sub["group"]=="treatment"]["conversions"].values[0]
    _, p, _, _, la, _ = two_proportion_ztest(cc, nc, ct, nt)
    sig = "Yes *" if p < ALPHA else "No"
    print(f"{g:<8} | {rc:>12.2%} | {rt:>14.2%} | {la:>+7.2%} | {sig}")

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(genders))
w = 0.35
for i, (group, color) in enumerate([("control","steelblue"),("treatment","coral")]):
    sub = gen_seg[gen_seg["group"]==group].sort_values("gender")
    ax.bar(x + i*w, sub["rate"]*100, width=w, label=group.title(), color=color, alpha=0.85)
ax.set_xticks(x + w/2)
ax.set_xticklabels(genders)
ax.set_ylabel("Conversion rate (%)")
ax.set_title("Conversion rate by gender")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Simpson's Paradox

Simpson's Paradox occurs when a trend present in several subgroups **reverses**
when the subgroups are combined. It arises when the subgroups have different sizes
and different baseline rates.

Our device_type data shows that Mobile, Desktop, and Tablet users all behave differently.
Below we construct a synthetic example using device type as the confounder to illustrate
how this paradox can emerge — and why always segmenting before concluding matters.

**Setup:** Mobile users convert at a lower rate than desktop users. The treatment group
happens to have more mobile users (a randomization imbalance in this dimension).
Aggregate results favor the control — but within each device type, treatment wins.

In [ ]:
# Synthetic Simpson's Paradox demonstration
np.random.seed(99)

# Desktop: high base rate, treatment wins
# Mobile:  low base rate, treatment wins
# But treatment group has more mobile users — aggregate reverses

simp = pd.DataFrame({
    "device":    ["desktop"]*40000 + ["mobile"]*10000 + ["desktop"]*20000 + ["mobile"]*30000,
    "group":     ["control"]*50000 + ["treatment"]*50000,
})
desktop_ctrl_rate, desktop_trt_rate = 0.20, 0.22
mobile_ctrl_rate,  mobile_trt_rate  = 0.05, 0.07

rates = {
    ("control",   "desktop"): desktop_ctrl_rate,
    ("control",   "mobile"):  mobile_ctrl_rate,
    ("treatment", "desktop"): desktop_trt_rate,
    ("treatment", "mobile"):  mobile_trt_rate,
}
simp["converted"] = simp.apply(
    lambda r: np.random.binomial(1, rates[(r.group, r.device)]), axis=1
)

# Aggregate result
agg = simp.groupby("group")["converted"].mean()
print("AGGREGATE (misleading):")
print(f"  Control:   {agg["control"]:.3%}")
print(f"  Treatment: {agg["treatment"]:.3%}")
print(f"  => Treatment looks WORSE by {agg["treatment"]-agg["control"]:+.3%}")

# Segment result
print()
print("BY DEVICE (correct):")
seg = simp.groupby(["device","group"])["converted"].mean().unstack()
for device in ["desktop", "mobile"]:
    print(f"  {device}: control={seg.loc[device,"control"]:.3%}, treatment={seg.loc[device,"treatment"]:.3%}, lift={seg.loc[device,"treatment"]-seg.loc[device,"control"]:+.3%}")
print(f"  => Treatment is BETTER in BOTH segments")
print()
print("Lesson: Always check whether aggregate results are driven by a confounding variable.")

In [ ]:
# Visualize the paradox
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Aggregate
ax = axes[0]
ax.bar(["Control","Treatment"], [agg["control"]*100, agg["treatment"]*100],
       color=["steelblue","coral"])
ax.set_title("Aggregate\n(misleading)", fontsize=11)
ax.set_ylabel("Conversion rate (%)")
ax.set_ylim(0, 25)

# Desktop
ax = axes[1]
ax.bar(["Control","Treatment"],
       [seg.loc["desktop","control"]*100, seg.loc["desktop","treatment"]*100],
       color=["steelblue","coral"])
ax.set_title("Desktop segment\n(treatment wins)", fontsize=11)
ax.set_ylim(0, 25)

# Mobile
ax = axes[2]
ax.bar(["Control","Treatment"],
       [seg.loc["mobile","control"]*100, seg.loc["mobile","treatment"]*100],
       color=["steelblue","coral"])
ax.set_title("Mobile segment\n(treatment wins)", fontsize=11)
ax.set_ylim(0, 25)

plt.suptitle("Simpson's Paradox: aggregate reverses segment-level truth", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Interaction Effects

A formal way to ask: "Does the treatment effect differ across segments?"

We test this by adding a **group × segment interaction term** and checking whether
the interaction coefficient is significant. A significant interaction means the effect
is **heterogeneous** — different users respond differently to the treatment.

In [ ]:
# Test for interaction between group and weekend/weekday
from scipy.stats import chi2_contingency

def segment_test(df, segment_col):
    results = []
    for seg_val in df[segment_col].unique():
        sub = df[df[segment_col] == seg_val]
        grp = sub.groupby("group")["converted"].agg(["sum","count"])
        if "control" not in grp.index or "treatment" not in grp.index:
            continue
        z, p, pc, pt, la, lr = two_proportion_ztest(
            grp.loc["control","sum"],   grp.loc["control","count"],
            grp.loc["treatment","sum"],  grp.loc["treatment","count"],
        )
        results.append({"segment": seg_val, "control_rate": pc,
                        "treatment_rate": pt, "lift": la, "p_value": p})
    return pd.DataFrame(results)

print("=== Weekday vs Weekend ===")
res = segment_test(df, "is_weekend")
res["segment"] = res["segment"].map({True: "Weekend", False: "Weekday"})
print(res.to_string(index=False))

print()
print("=== By Day of Week ===")
res2 = segment_test(df, "weekday")
print(res2.sort_values("p_value").to_string(index=False))

## 5. Guardrails for Segment Analysis

**Pre-specify your segments.** Running post-hoc segment searches inflates false positive
rates — with 20 segments, you expect 1 significant result by chance at α=0.05.

| Type | Description | How to handle |
|---|---|---|
| Pre-specified segments | Defined in experiment plan before launch | Test at nominal α |
| Exploratory segments | Discovered by looking at the data | Adjust α (see notebook 06) |
| Guardrail segments | Groups who must not be harmed | One-tailed test, strict α |

**Segment findings are hypothesis-generating, not hypothesis-confirming.**
A surprising segment result should lead to a new, dedicated experiment for that segment.

Proceed to **[07 — Multiple Testing](07_multiple_testing.ipynb)** for correction methods.